### Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Connecting to the SQL Database

In [ ]:
import sqlite3
connection = sqlite3.connect(r"C:\Users\cmdvi\desktop\Shift_Optimization_System\ShiftData.db")
Shift_Data = pd.read_sql("SELECT * FROM ShiftPerformance", connection)
Shift_Data.head()

### Basic Dataset Overview
* Shift_Data.columns - list the column names in the dataset
* Shift_Data.shape - getting the shape of the dataset
* Format the data to proper date format
* df.duplicated().sum() - checking for duplicates in our dataset
* df.isna().sum() - checking for missing values in the dataset
* df.describe() - checking the statistical distribution of our dataset


In [ ]:
Shift_Data.columns

In [ ]:
Shift_Data.shape

In [ ]:
Shift_Data.info()

##### Properly formatting the columns that has the datatype of data into proper datetime format

In [ ]:
Shift_Data['date'] = pd.to_datetime(Shift_Data['date'])
Shift_Data['start_time'] = pd.to_datetime(Shift_Data['start_time'])
Shift_Data['end_time'] = pd.to_datetime(Shift_Data['end_time'])
Shift_Data['timestamp'] = pd.to_datetime(Shift_Data['timestamp'])

In [ ]:
Shift_Data.info()

In [ ]:
Shift_Data.duplicated().sum()

In [ ]:
Shift_Data.isna().sum()

In [ ]:
## This cell checks the statistical distribution of our dataset
Shift_Data.describe()

### Filling the missing values
* df.isna().sum() - identify the columns that has the missing values and show how many values are missing
* df.sort_values(by='date') - sort the dataset chronologically, which is essential for timeseries before using the forward of filling missing values
* Filling the missing values with the last known values, and any other gap that remains is filled with mean

In [ ]:
Shift_Data = Shift_Data.sort_values(by='date')

# fill temperature and Humidity with mean values
Shift_Data['temperature'] = Shift_Data['temperature'].fillna(method="ffill").fillna(Shift_Data['temperature'].mean())
Shift_Data['humidity'] = Shift_Data['humidity'].fillna(method="ffill").fillna(Shift_Data['humidity'].mean())

# fill timestamp
Shift_Data['timestamp'] = Shift_Data['timestamp'].fillna(method="ffill")


In [ ]:
Shift_Data.isna().sum()

In [ ]:
# Fill the categorical fields

Shift_Data['issue_type'] = Shift_Data['issue_type'].fillna('No Issue')
Shift_Data['resolved_by'] = Shift_Data['resolved_by'].fillna('No Maintenance')

# Fill the maintenance downtime with 0 for shifts without issues
Shift_Data['maintenance_downtime'] = Shift_Data['maintenance_downtime'].fillna(0)

Shift_Data = Shift_Data.drop(columns=['maintenance_id'])

In [ ]:
Shift_Data.isna().sum()

## Exploratory Data Analysis


**Numerical Variables Analysis**
- Created histograms to visualize the distribution of all numerical columns (units produced, defects, cycle time, efficiency, runtime, downtime, temperature, humidity)
- Generated boxplots to identify outliers and understand data spread for each numerical variable

**Categorical Variables Analysis**
- Displayed value counts for all categorical columns (shift name, experience level, skill category, machine
status, issue type, defect type, severity, inspection result)
- Created bar charts in a 4x2 grid layout to visualize the frequency distribution of each categorical variable

**Maintenance Flag Analysis**
- Grouped numerical metrics by maintenance flag (0 = no maintenance, 1 = maintenance performed)
- Created grouped bar charts comparing all metrics between the two maintenance flag categories

**Shift Performance Analysis**
- Aggregated key performance indicators by shift (average units, total units, cycle time, defects, downtime,
efficiency)
- Calculated defect rate for each shift (total defects + total units produced)
- Visualized shift KPIs using bar charts in a 2x3 grid layout

**Productivity Metrics**
- Created a new feature: total machine hours (runtime hpurs + downtime minutes converted to hours)
Calculated output per hour for each shift (total units produced + total machine hours)
. Compared output per hour against average efficiency scores across shifts

**Availability Analysis**
- Calculated machine availability for each shift (runtime hours + total available time)
- Availability represents the percentage of time machines were actually running versus total scheduled production
time
- Visualized availability by shift with percentage labels

---

In [ ]:
## Numerical Data Analysis

num_cols = [
   "units_produced","defect_count","cycle_time_avg",
   "shift_efficiency_score", "runtime_hours", "downtime_minutes", "maintenance_downtime",
   "temperature", "humidity"
]

Shift_Data[num_cols].hist(figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
## Using boxplots to identify outliers in numerical features
plt.figure(figsize=(15, 10))
for i, col in enumerate(num_cols):
    plt.subplot(3, 3, i+1)
    sns.boxplot(y=Shift_Data[col])
    plt.title(col)

plt.tight_layout()
plt.show()

## Categorical Variables Analysis

In [ ]:
Categorical_cols = ["shift_name", "issue_type", "resolved_by",
                    "machine_status", "defect_type", "inspection_result",
                    "severity", "experience_level"]
for column in Categorical_cols:
    print(f"Value counts for {column}:\n")
    print(Shift_Data[column].value_counts())

   

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 20))

axes = axes.flatten()

for i, column in enumerate(Categorical_cols):
    # Get the value counts for the column
    counts = Shift_Data[column].value_counts().reset_index()
    counts.columns = [column, 'count']

    sns.barplot(data=counts, x=column, y='count', palette = 'coolwarm', ax=axes[i])

   # plot customization
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_title(f"Distribution for {column}", fontsize=14, fontweight='bold')
    axes[i].set_xlabel(column)
    axes[i].set_ylabel('Count')

    # add the value labels on the bars
    for j, v in enumerate(counts['count']):
        axes[i].text(j, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
                     
plt.tight_layout()
plt.show()



### Operational Performance during Maintenance vs No Maintenance

In [ ]:
Shift_Data["maintenance_flag"].value_counts()

In [ ]:
# Maintenance Flags Analysis
maintenance_data = Shift_Data.groupby('maintenance_flag')[num_cols].mean()

maintenance_data.head()


In [ ]:
maintenance_plot = maintenance_data.reset_index()
maintenance_plot_melt = maintenance_plot.melt(
    id_vars=['maintenance_flag'], 
    var_name='performance', 
    value_name='value'
    )

plt.figure(figsize=(14, 6))
sns.barplot(
    data = maintenance_plot_melt,
    x = 'performance',
    y = 'value',
    hue = 'maintenance_flag',
    palette = 'Set2'
)
plt.xticks(rotation=45, ha='right')
plt.title('Operational Performance based on Maintenance Flag')
plt.legend(title = 'Maintenance Flag')
plt.tight_layout()


### Shift Performance Analysis

In [ ]:
shift_performance_data = Shift_Data.groupby('shift_name').agg({
    'units_produced': ['mean', 'sum'],
    'defect_count': 'mean',
    'cycle_time_avg': 'mean',
    'shift_efficiency_score': 'mean',
    'downtime_minutes': ['mean', 'sum']
}).reset_index()

shift_performance_data.columns = [
    'shift_name',
    'average_units_produced',
    'total_units_produced',
    'average_defect_count',
    'average_cycle_time',
    'average_efficiency',
    'average_downtime',
    'total_downtime'
]

# calculate the defect rate
defect_sum = Shift_Data.groupby('shift_name')['defect_count'].sum().values
units_sum = Shift_Data.groupby('shift_name')['units_produced'].sum().values

shift_performance_data['defect_rate'] = (defect_sum / units_sum) * 100  # Convert to percentage

shift_performance_data.sort_values(by= 'average_units_produced', ascending = False)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 10))

axes = axes.flatten()

metrics = ['average_units_produced', 'total_units_produced', 'average_cycle_time', 'average_defect_count', 'average_downtime', 'total_downtime', 'defect_rate', 'average_efficiency']

for i, metric in enumerate(metrics):
    sns.barplot(data = shift_performance_data,
                x = 'shift_name',
                y = metric,
                palette = 'Set2',
                legend = False,
                ax = axes[i])
    axes[i].set_title(f'{metric.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Shift Name')
    axes[i].set_ylabel(metric.replace("_", " ").title())

    for j, v in enumerate(shift_performance_data[metric]):
        if 'units' in metric:
            axes[i].text(j, v, f'{v:,.0f}', ha='center', va='bottom', fontsize = 9)
        else:
            axes[i].text(j, v, f'{v:,.2f}', ha='center', va='bottom', fontsize = 9)
            
plt.suptitle('Shift Performance across board', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### Shift Performance based on production output per hour

In [ ]:
Shift_Data["total_operation_hours"] = Shift_Data['runtime_hours'] + (Shift_Data['downtime_minutes'] / 60)

output_per_hour = (
    Shift_Data.groupby('shift_name')
    .apply(lambda x:x['units_produced'].sum() / x['total_operation_hours'].sum())
    .rename('output_per_hour')
)
output_per_hour

In [ ]:
output_per_hour_data = output_per_hour.reset_index()

fig,axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(
    data = output_per_hour_data,
    x = 'shift_name',
    y = 'output_per_hour',
    hue = 'shift_name',
    palette = 'Set2',
    ax = axes[0],
    legend = False
)

axes[0].set_title('Output per Hour by Shift', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Shift Name')
axes[0].set_ylabel('Unit Produced per hour')

for i, v in enumerate(output_per_hour_data['output_per_hour']):
    axes[0].text(i, v, f'{v:.2f}', ha='center', va='bottom')

merged_data = shift_performance_data.merge(output_per_hour_data, on='shift_name')
# Plot 2, plotting the shift efficiency score 

sns.barplot(
    data = merged_data,
    x = 'shift_name',
    y = 'average_efficiency',
    hue = 'shift_name',
    palette = 'Set2',
    ax = axes[1],
    legend = False
)

axes[1].set_title('Efficiency Rate by Shift', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Shift Name')
axes[1].set_ylabel('Average Efficiency score')

for i, v in enumerate(merged_data['average_efficiency']):
    axes[1].text(i, v, f'{v:.2f}', ha='center', va='bottom')

plt.suptitle('Shift Output Per Hour and Efficiency Score', fontsize = 16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
production = (
    Shift_Data[['production_id', 'shift_id', 'date', 'units_produced', 'defect_count']]
    .drop_duplicates(subset = ['production_id'])
    .copy()
)

machine = (
    Shift_Data[['machine_id', 'date', 'shift_id', 'downtime_minutes']]
    .drop_duplicates()
    .copy()
)

In [ ]:
production.head()

In [ ]:
production.shape

In [ ]:
machine.head()

In [ ]:
machine.shape

### OEE Metrics Calculation

In [ ]:
import os

PLANNED = globals().get('PLANNED_HOURS', 7.5)
THEO = globals().get('THEO_RATE', 100.0)

### Compute per machine availability
machine['available_machine'] = 1 - machine['downtime_minutes'] / (PLANNED * 60)
machine['available_machine'] = machine['available_machine'].clip(0, 1)

available_machine_agg =(
    machine.groupby(['date', 'shift_id'])['available_machine']
    .mean()
    .reset_index()
    .rename(columns={'available_machine': 'machine_availability'})
)

In [ ]:
available_machine_agg.head()

In [ ]:
available_machine_agg.shape

In [ ]:
# merging availability into production
merged_production_data = production.merge(available_machine_agg, on=['date', 'shift_id'], how='left')
merged_production_data['machine_availability'] = merged_production_data['machine_availability'].fillna(1)

# performance based on the planned time (not runtime)
merged_production_data['performance'] = (merged_production_data['units_produced'] / (THEO * PLANNED)).clip(0, 1)

# Quality
merged_production_data['quality'] = 1 - (merged_production_data['defect_count'] / merged_production_data['units_produced'])
merged_production_data['quality'] = merged_production_data['quality'].fillna(1).clip(0, 1)

# Final OEE estimate
merged_production_data['oee_est'] = merged_production_data['machine_availability']  * merged_production_data['performance'] * merged_production_data['quality']

merged_production_data['month'] = merged_production_data['date'].dt.month


In [ ]:
merged_production_data.head()

In [ ]:
## Monthly OEE (Overall Equipment Effectiveness for each month)
monthly_oee = (
    merged_production_data.groupby('month')['oee_est']
    .mean()
    .sort_index()
)

# Print to confirm it matches the earlier value
for month, value in monthly_oee.items():
    print(f'Month {month} OEE: {value*100:.2f}%')

In [ ]:
if 1 in monthly_oee.index and 3 in monthly_oee.index:
    m1 = monthly_oee.loc[1]
    m3 = monthly_oee.loc[3]
    degradation = (m1 - m3) / m1 * 100
    print(f'Quarterly degradation (month 1 -> month 3): {degradation:.2f}')
else:
    print('could not find month 1 and 3 in the data to compute the degradation process or curve')

In [ ]:
monthly_oee_df = monthly_oee.reset_index()
monthly_oee_df['OEE_percentage'] = monthly_oee_df['oee_est'] * 100

sns.lineplot(x='month', y ='OEE_percentage', data = monthly_oee_df, marker = 'o')
plt.title('Monthly OEE Trend')
plt.ylabel('OEE %')
plt.ylim(60, 80)
plt.xticks([1, 2, 3])
plt.xlabel('Month')
plt.show()

### Experience vs the defect counts that occured during production

In [ ]:
sns.boxplot(data = Shift_Data, x ='experience_level', y ='defect_count', palette = 'Set2')
plt.title('Operator Experience vs Defect Count')
plt.xlabel('Experience (Year)')
plt.ylabel('Defect Count')
plt.show()

## Average Throughput by shift
* Measures how many units are produced within a given time
* Throughput = Units Produced / Runtime hours

In [ ]:
Throughput= (
    Shift_Data['units_produced'] /
    Shift_Data['runtime_hours']
)

In [ ]:
avg_throughput = (
    Shift_Data.groupby('shift_name')
    .apply(
        lambda x: (
            x['units_produced'] / x['runtime_hours']
        ).mean()
    )
    .reset_index(name='avg_throughput')
)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=avg_throughput,
    x='shift_name',
    y='avg_throughput',
    hue= 'shift_name',
    palette='Set2'
)


plt.title('Average Throughput by Shift')
plt.xlabel('Shift Name')
plt.ylabel('Average Throughput\n(Units per Runtime Hour)')

plt.show()

### Correlation heat map

In [ ]:
numeric_columns = [
    'units_produced',
    'defect_count',
    'cycle_time_avg',
    'runtime_hours',
    'downtime_minutes',
    'temperature',
    'humidity',
    'shift_efficiency_score',
    'maintenance_downtime',
    'experience_level'
]

corr_matrix = Shift_Data[numeric_columns].corr()

plt.figure(figsize=(12,8))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm'
)

plt.title('Correlation Heatmap')

plt.show()

### Feature Engineering

In [ ]:
Shift_Data.info()

In [ ]:
# Engineering new features
import pandas as pd

Shift_Data['start_time'] = pd.to_datetime(Shift_Data['start_time'])
Shift_Data['end_time'] = pd.to_datetime(Shift_Data['end_time'])
Shift_Data['date'] = pd.to_datetime(Shift_Data['date'])

# fixing the overnight shift and add 1 day where the end time < start time
mask = Shift_Data['end_time'] < Shift_Data['start_time']
Shift_Data.loc[mask, 'end_time'] = Shift_Data.loc[mask, 'end_time'] + pd.Timedelta(days=1)

# Shift duration
Shift_Data['shift_duration'] = (Shift_Data['end_time'] - Shift_Data['start_time']).dt.total_seconds() / 3600

# Defect rate
Shift_Data['defect_rate'] = Shift_Data['defect_count'] / Shift_Data['units_produced'].replace(0, pd.NA)

Shift_Data['downtime_ratio'] = Shift_Data['downtime_minutes'] / (Shift_Data['shift_duration'] * 60)

# temporal feature
Shift_Data['day_of_week'] = Shift_Data['date'].dt.dayofweek
Shift_Data['hour_of_day'] = Shift_Data['start_time'].dt.hour

In [ ]:
Shift_Data.shape

## Columns dropped
* Identifiers- not neccessary for the modeling phase
* defect rate - may cause redundancy since it was derived— leakage
* downtime ratio - may cause redundancy since it was derived — leakage
* total_operation_hours - may cause redundancy since it was derived — leakage
* inspection result - it is an outcome of the shift/maintenance, not a predictor of the shift efficiency
* datetime columns- not needed, adds little value

In [ ]:
# drop identifiers and leakage, raw datetime columns

columns_to_drop = [
    #identifier
    'shift_id', 'operator_id', 'operator_name', 'machine_id', 'production_id', 'supervisor_id','qc_id', 'resolved_by',
    #leakage
    'inspection_result', 'defect_rate', 'downtime_ratio', 'total_operation_hours',
    #Raw datetime columns
    'start_time', 'end_time', 'date', 'timestamp', 'day_of_week', 'hour_of_day'
]

Shift_Data.drop(columns = columns_to_drop, inplace = True)

In [ ]:
Shift_Data.columns

In [ ]:
Shift_Data.duplicated().sum()

In [ ]:
duplicates = Shift_Data[Shift_Data.duplicated(keep=False)]
print(duplicates.head(20))

In [ ]:
duplicate_count = Shift_Data.duplicated().sum()

duplicate_pct = (
    duplicate_count / len(Shift_Data)
) * 100

print(f"{duplicate_pct:.2f}% duplicates")

In [ ]:
Shift_Data = Shift_Data.drop_duplicates()

In [ ]:
Shift_Data.duplicated().sum()

## Modeling Phase

In [ ]:
Shift_Data.isna().sum()

In [ ]:
Shift_Data.head()

In [ ]:
Shift_Data['shift_efficiency_score'].describe()

## Data Splitting

In [ ]:
X = Shift_Data.drop(columns=['shift_efficiency_score'])
y = Shift_Data['shift_efficiency_score']

In [ ]:
X.head()

In [ ]:
## Splitting the data into training and testing

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

Categorical_features = [
    'shift_name',
    'skill_category',
    'machine_status',
    'issue_type',
    'defect_type',
    'severity'
]

Numerical_features = [
    'units_produced',
    'cycle_time_avg',
    'experience_level',
    'defect_count',
    'runtime_hours',
    'downtime_minutes', 
    'maintenance_flag',
    'maintenance_downtime',
    'temperature',
    'humidity',
    'shift_duration'
]

preprocess = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(handle_unknown ='ignore'), Categorical_features),
        ('num', 'passthrough', Numerical_features)
    ]
)

In [ ]:
import dagshub
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import dagshub
dagshub.init(repo_owner='oyemi21', repo_name='Shift_Optimization_System', mlflow=True)


models = {
    "LinearRegressionModel": LinearRegression(),
    "RandomForest": RandomForestRegressor(),
    "GradientBoosting": GradientBoostingRegressor()
}

results = {}
fitted_pipeline = {}

mlflow.set_experiment("Nordex_Shift_Optimization_models")

for name, model in models.items():

    with mlflow.start_run(run_name = name):
        # creating the pipeline
        pipeline = Pipeline(steps=[
            ('pre_process', preprocess),
            ('model', model)
        ])

        # training the models on the train data
        pipeline.fit(X_train, y_train)

        # getting the model to predict the test data
        y_pred = pipeline.predict(X_test)

        # Defining Metrics
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)

        results[name] = {"r2": r2, "mae": mae, "mse": mse}

        fitted_pipeline[name] = pipeline

        # Mlflow logging
        mlflow.log_param("model_name", name)

        if hasattr(model, "get_params"):
            for param_key, param_value in model.get_params().items():
                mlflow.log_param(param_key, param_value)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mse", mse)

        mlflow.sklearn.log_model(pipeline, artifact_path="model")

        print(f"{name} -> R2: {r2:.4f}, MAE: {mae:.4f}, MSE: {mse:.4f}")